In [ ]:
# =============================================================================
# CTRL ALT ELITE — ENGG2112 Final Project
# Notebook 2: Model Pipeline
# Flight Delay Prediction at Chicago O'Hare (ORD)
# Models: Logistic Regression | Random Forest | XGBoost | MLP
#
# PREREQUISITE: Run Notebook 1 (Data Cleaning) first.
# Required input file: ord_flights_weather_calendar.csv
# =============================================================================
#
# HOW TO USE THIS NOTEBOOK
# ------------------------
# 1. Run Notebook 1 (Data Cleaning) first to generate:
#       ord_flights_weather_calendar.csv
# 2. Place that CSV in the same folder as this notebook.
# 3. Install any missing libraries:
#       pip install xgboost imbalanced-learn scikit-learn pandas matplotlib seaborn statsmodels
# 4. Run all cells from top to bottom.
#
# The script will print results to the terminal and save all plots
# as .png files in the same folder.
# =============================================================================


# =============================================================================
# STEP 0: IMPORTS
# =============================================================================
# We import everything we need up front so it's easy to spot missing libraries.
#
# - pandas / numpy: data loading and manipulation
# - sklearn:        our ML models (LR, RF, MLP) plus all evaluation tools
# - xgboost:        XGBoost classifier (installed separately, not part of sklearn)
# - imblearn:       SMOTE for oversampling the minority (delayed) class
# - matplotlib / seaborn: all visualisations
# =============================================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, accuracy_score,
    precision_score, recall_score, f1_score
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
np.random.seed(42)

# Set a clean visual style for all plots
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
PLOT_DIR = "."   # change this if you want plots saved to a subfolder
PRECISION_TARGET = 0.50  # minimum acceptable precision for passenger-facing delay alerts
                         # 0.50 = at least 1 in 2 alerts is a genuine delay


# =============================================================================
# STEP 1: LOAD DATA
# =============================================================================
# We load the three-dataset merged file that was produced during data cleaning.
# This file contains 81,452 ORD flights (2019, 2022, 2023) with weather
# and calendar features already joined in.
#
# We run a quick sense-check to confirm the shape and class balance before
# doing anything else. Always verify your data is what you expect it to be.
# =============================================================================

print("=" * 65)
print("STEP 1: LOADING DATA")
print("=" * 65)

df = pd.read_csv("ord_flights_weather_calendar.csv", low_memory=False)

print(f"  Dataset shape:       {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Years present:       {sorted(df['year'].unique())}")
print(f"  Delayed flights:     {df['is_delayed'].sum():,}  ({df['is_delayed'].mean()*100:.1f}%)")
print(f"  On-time flights:     {(~df['is_delayed'].astype(bool)).sum():,}  ({(1-df['is_delayed'].mean())*100:.1f}%)")
print()


# =============================================================================
# STEP 2: DEFINE FEATURES AND REMOVE LEAKAGE COLUMNS
# =============================================================================
# This is the most critical methodological decision in the whole pipeline.
#
# DATA LEAKAGE means using information that would not be available at
# prediction time — i.e., information that is only known AFTER the flight
# has departed or arrived.
#
# If we included columns like ARR_DELAY or TAXI_OUT, the model would
# "cheat" by seeing information from the future. It would score perfectly
# on training data but be completely useless in the real world.
# The Kaggle notebook we reviewed earlier made exactly this mistake and
# achieved accuracy = 1.0, which is the hallmark of leakage.
#
# COLUMNS WE DROP (post-departure / post-arrival data):
#   DEP_TIME          — actual departure time (only known after departure)
#   TAXI_OUT          — time on taxiway (only known after wheels-off)
#   WHEELS_OFF        — wheels-off time
#   WHEELS_ON         — wheels-on time (arrival)
#   TAXI_IN           — taxi time on arrival
#   ARR_TIME          — actual arrival time
#   ARR_DELAY         — arrival delay (directly correlated with dep delay)
#   ELAPSED_TIME      — actual flight duration
#   AIR_TIME          — time in the air
#   DELAY_DUE_*       — the CAUSE of the delay (only known after it happens)
#   CANCELLED         — only known after scheduled departure time
#   DEP_DELAY         — our regression target, not a classification feature
#   is_delayed        — our classification TARGET (not a feature!)
#   FL_DATE           — raw date string, we already have month/day_of_week etc.
#   ORIGIN            — always 'ORD', zero variance, adds nothing
#   holiday_name      — free text, redundant with is_federal_holiday flag
#
# COLUMNS WE KEEP:
#   All temporal features engineered from scheduled departure time
#   All weather features (known the morning of the flight)
#   All calendar features (known in advance)
#   AIRLINE_CODE, DEST — known at booking time
#   DISTANCE, CRS_DEP_TIME — known at booking time
# =============================================================================

print("=" * 65)
print("STEP 2: DEFINING FEATURES AND REMOVING LEAKAGE")
print("=" * 65)

# Columns that would cause data leakage — known only after the flight operates
LEAKAGE_COLS = [
    "DEP_TIME", "TAXI_OUT", "WHEELS_OFF", "WHEELS_ON",
    "TAXI_IN", "ARR_TIME", "ARR_DELAY", "ELAPSED_TIME", "AIR_TIME",
    "DELAY_DUE_CARRIER", "DELAY_DUE_WEATHER", "DELAY_DUE_NAS",
    "DELAY_DUE_SECURITY", "DELAY_DUE_LATE_AIRCRAFT",
    "CANCELLED",
]

# Identifier / target / redundant columns to also exclude
NON_FEATURE_COLS = [
    "DEP_DELAY",       # regression target — keep aside, don't use as classifier feature
    "is_delayed",      # classification target — keep aside
    "FL_DATE",         # raw date string, superseded by month/day_of_week etc.
    "ORIGIN",          # always 'ORD', zero variance
    "holiday_name",    # free text, redundant with is_federal_holiday
    "STATION", "NAME", # weather station identifiers, not predictive
]

# The two targets we will use
TARGET_CLF = "is_delayed"     # binary: 0 = on time, 1 = delayed ≥ 15 min
TARGET_REG = "DEP_DELAY"      # continuous: departure delay in minutes

# Drop duplicated season column if it exists (artefact of the three-way merge)
if "season_x" in df.columns:
    df = df.rename(columns={"season_x": "season"})
if "season_y" in df.columns:
    df = df.drop(columns=["season_y"])

# Build the feature set: all columns except leakage, non-features, and targets
ALL_EXCLUDE = set(LEAKAGE_COLS + NON_FEATURE_COLS)
feature_cols = [c for c in df.columns if c not in ALL_EXCLUDE]

print(f"  Columns excluded (leakage + identifiers): {len(ALL_EXCLUDE)}")
print(f"  Feature columns remaining:                {len(feature_cols)}")
print(f"  (These will be reduced to 10 after multicollinearity analysis in Step 2b)")
print()

# --------------------------------------------------------------------------
# ENCODE CATEGORICAL COLUMNS
# --------------------------------------------------------------------------
# Tree-based models (RF, XGBoost) can technically handle categories, but
# sklearn and xgboost both require numeric input. We use Label Encoding:
# each unique category gets a unique integer.
#
# FIX: We use pd.to_numeric() to detect non-numeric columns rather than
# relying on dtype == "object". This is more robust because some string
# columns may be stored with a non-object dtype depending on how the CSV
# was saved and loaded (e.g. mixed-type columns, pandas version differences).
# pd.to_numeric(..., errors='coerce') converts whatever it can to a number
# and turns anything it can't convert into NaN. If a column gains any NaN
# values through this process, it must have contained strings — so we encode
# it with LabelEncoder instead.
# --------------------------------------------------------------------------

le = LabelEncoder()
cat_cols = []

for col in feature_cols:
    # Attempt numeric conversion — non-numeric values become NaN
    converted = pd.to_numeric(df[col], errors="coerce")
    if converted.isnull().any() and not df[col].isnull().all():
        # This column contains strings that couldn't be converted → encode it
        cat_cols.append(col)
        df[col] = df[col].fillna("Unknown").astype(str)
        df[col] = le.fit_transform(df[col])
    else:
        # Fully numeric — just ensure the column is stored as float
        df[col] = converted

print(f"  Categorical columns detected and encoded: {cat_cols}")

# Fill any remaining numeric nulls with the column median
num_cols = [c for c in feature_cols if c not in cat_cols]
for col in num_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

print(f"  Null values remaining after encoding: {df[feature_cols].isnull().sum().sum()}")
print()




# =============================================================================
# STEP 2b: MULTICOLLINEARITY ASSESSMENT AND FEATURE SELECTION
# =============================================================================
# Before training, we assess multicollinearity — the degree to which features
# are correlated with each other. Highly correlated features cause three problems:
#
#   1. REDUNDANCY: Two correlated features carry the same information, so one
#      of them adds no predictive value.
#   2. MODEL INSTABILITY: Logistic Regression and MLP coefficients become
#      unreliable when features are collinear — small changes in the data cause
#      large swings in weights.
#   3. INTERPRETABILITY: Feature importance scores are split across correlated
#      variables, making it harder to understand what the model is actually
#      learning.
#
# We use two complementary tools:
#
#   A. CORRELATION MATRIX (Pearson r):
#      A value of |r| > 0.70 between two features signals high collinearity.
#      We visualise the full matrix as a heatmap to identify clusters of
#      correlated features (e.g. TMIN / TMAX / avg_temp are all temperature
#      measurements and will be highly correlated with each other).
#
#   B. VARIANCE INFLATION FACTOR (VIF):
#      VIF measures how much the variance of a feature's regression coefficient
#      is inflated due to collinearity with all OTHER features combined.
#      Rule of thumb: VIF > 10 = problematic; VIF > 5 = worth investigating.
#      VIF = 1 means no collinearity with any other feature.
#
# Based on both tools, we manually select 10 final features that:
#   - Are among the most predictive (from feature importance in the literature
#     and from our domain understanding of flight delays)
#   - Are NOT highly correlated with each other (|r| < 0.70 between any pair)
#   - Each represent a distinct real-world concept
# =============================================================================

print("=" * 65)
print("STEP 2b: MULTICOLLINEARITY ASSESSMENT AND FEATURE SELECTION")
print("=" * 65)

from statsmodels.stats.outliers_influence import variance_inflation_factor

# ── A. CORRELATION MATRIX ──────────────────────────────────────────────────
# Compute pairwise Pearson correlation across all 58 features.
# We only use numeric features for this analysis.
numeric_feature_cols = [c for c in feature_cols if df[c].dtype in [float, int]
                        or pd.api.types.is_numeric_dtype(df[c])]

corr_matrix = df[numeric_feature_cols].corr().abs()

# Plot the full correlation heatmap
fig, ax = plt.subplots(figsize=(20, 16))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # show lower triangle only
sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap="coolwarm",
    center=0,
    vmin=0, vmax=1,
    linewidths=0.3,
    ax=ax,
    cbar_kws={"shrink": 0.6, "label": "|Pearson r|"},
    annot=False,
    xticklabels=True, yticklabels=True
)
ax.set_title("Feature Correlation Matrix (|Pearson r|) — All 58 Features",
             fontsize=14, fontweight="bold", pad=15)
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0,  fontsize=7)
plt.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, "correlation_matrix.png"), dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: correlation_matrix.png")

# Print the most collinear pairs (|r| > 0.70)
print("\n  Highly correlated feature pairs (|r| > 0.70):")
high_corr_pairs = []
cols = corr_matrix.columns.tolist()
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        r = corr_matrix.iloc[i, j]
        if r > 0.70:
            high_corr_pairs.append((cols[i], cols[j], r))

high_corr_pairs.sort(key=lambda x: -x[2])
for a, b, r in high_corr_pairs[:20]:
    print(f"    |r| = {r:.3f}   {a}  ↔  {b}")
print()

# ── B. VIF ANALYSIS ────────────────────────────────────────────────────────
# VIF requires a clean numeric matrix with no nulls or near-zero-variance cols.
# We drop any column where all values are identical (VIF would be undefined).
vif_cols = [c for c in numeric_feature_cols
            if df[c].nunique() > 1 and not df[c].isnull().any()]

vif_data = df[vif_cols].copy().astype(float)
vif_scores = {}
for i, col in enumerate(vif_cols):
    try:
        vif_scores[col] = variance_inflation_factor(vif_data.values, i)
    except Exception:
        vif_scores[col] = float("nan")

vif_df = pd.Series(vif_scores).sort_values(ascending=False)
print("  Top 20 features by VIF (higher = more collinear with others):")
print(f"  {'Feature':<30} {'VIF':>8}")
print(f"  {'-'*40}")
for feat, v in vif_df.head(20).items():
    flag = "  ← HIGH" if v > 10 else ("  ← moderate" if v > 5 else "")
    print(f"  {feat:<30} {v:>8.1f}{flag}")
print()

# ── C. FINAL FEATURE SELECTION ─────────────────────────────────────────────
# Based on the correlation matrix and VIF analysis above, we select 10 features
# that each represent a DISTINCT real-world concept and are not highly
# correlated with each other:
#
#  Feature            | Concept             | Why kept / why alternatives dropped
#  ───────────────────┼─────────────────────┼──────────────────────────────────────
#  PRCP               | Precipitation       | Strongest RF predictor; continuous
#                     |                     | has_precip dropped (|r|≈0.85 with PRCP)
#  AWND               | Wind speed          | Distinct weather axis from precipitation
#  SNOW               | Snowfall            | Adds snow-specific signal beyond PRCP
#                     |                     | SNWD & has_snow_cover dropped (redundant)
#  avg_temp           | Temperature         | TMIN and TMAX dropped (|r|>0.95 with each other
#                     |                     | and with avg_temp)
#  CRS_DEP_TIME       | Time of day         | Peak-hour congestion proxy
#  day_of_year        | Seasonality         | Encodes weather seasons continuously
#                     |                     | month & quarter dropped (|r|>0.90 with day_of_year)
#  is_weekend         | Day type            | Lower traffic pattern on weekends vs weekdays
#  daily_dep_count    | Airport congestion  | Unique operational load signal
#  DISTANCE           | Route length        | Longer routes have different delay profiles
#                     |                     | CRS_ELAPSED_TIME dropped (|r|≈0.99 with DISTANCE)
#  AIRLINE_CODE       | Carrier             | On-time performance varies significantly by airline

FINAL_FEATURES = [
    "PRCP",             # precipitation (mm) — top RF feature, strong delay predictor
    "AWND",             # average wind speed — independent weather axis
    "SNOW",             # snowfall (mm) — distinct from rain; major delay cause at ORD
    "avg_temp",         # average daily temperature — TMIN/TMAX are redundant proxies
    "CRS_DEP_TIME",     # scheduled departure time — captures peak-hour congestion
    "day_of_year",      # day of year (1–366) — captures seasonality continuously
    "is_weekend",       # binary: weekend vs weekday — traffic volume proxy
    "daily_dep_count",  # total departures from ORD that day — congestion indicator
    "DISTANCE",         # route distance (miles) — longer routes buffer delays differently
    "AIRLINE_CODE",     # encoded carrier — on-time performance differs by airline
]

# Verify all selected features exist in our encoded dataframe
missing = [f for f in FINAL_FEATURES if f not in df.columns]
if missing:
    print(f"  WARNING: These selected features not found in dataframe: {missing}")
else:
    print(f"  ✓ All 10 selected features present in dataset")

# Replace the full feature list with the curated 10
feature_cols = FINAL_FEATURES

print(f"\n  Final feature set ({len(feature_cols)} features):")
for i, f in enumerate(feature_cols, 1):
    print(f"    {i:2d}. {f}")
print()

# Plot a focused correlation heatmap for just the 10 selected features
corr_10 = df[FINAL_FEATURES].corr().abs()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr_10,
    annot=True, fmt=".2f",
    cmap="coolwarm",
    center=0, vmin=0, vmax=1,
    linewidths=0.5,
    ax=ax,
    cbar_kws={"shrink": 0.8, "label": "|Pearson r|"}
)
ax.set_title("Correlation Matrix — Final 10 Selected Features",
             fontsize=13, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, "correlation_matrix_final10.png"), dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: correlation_matrix_final10.png")
print()

# =============================================================================
# STEP 3: TEMPORAL TRAIN / TEST SPLIT
# =============================================================================
# We use a TEMPORAL split rather than a random split. This is the correct
# approach whenever your data has a time dimension and you want your model
# to reflect real-world deployment conditions.
#
# WHY NOT RANDOM SPLIT?
# A random 80/20 split would scatter 2023 flights throughout the training
# set. The model would "remember" future flights during training, inflating
# test accuracy and hiding any temporal drift in airline operations, airport
# infrastructure, or weather patterns. This is sometimes called "temporal
# data leakage".
#
# OUR SPLIT:
#   Train: 2019 flights + Jan–Feb 2020 flights + 2022 flights
#          (~64,000 flights, ~79% of data)
#   Test:  2023 flights only
#          (~17,000 flights, ~21% of data)
#
# We exclude March 2020 – December 2021 because that period covers the
# COVID-19 pandemic, when government-supported ghost flights, near-zero
# passenger demand, and suppressed delay rates represent a structural break
# in normal airport operations. Including this data would teach the model
# patterns that do not apply to normal operations.
# =============================================================================

print("=" * 65)
print("STEP 3: TEMPORAL TRAIN / TEST SPLIT")
print("=" * 65)

TEST_YEAR = 2023

train_mask = df["year"] != TEST_YEAR
test_mask  = df["year"] == TEST_YEAR

X_train = df.loc[train_mask, feature_cols].copy()
X_test  = df.loc[test_mask,  feature_cols].copy()
y_train = df.loc[train_mask, TARGET_CLF].values
y_test  = df.loc[test_mask,  TARGET_CLF].values

# Also keep regression targets for the regression step at the end
y_reg_train = df.loc[train_mask, TARGET_REG].values
y_reg_test  = df.loc[test_mask,  TARGET_REG].values

print(f"  Training set:  {X_train.shape[0]:,} flights   "
      f"(delay rate: {y_train.mean()*100:.1f}%)")
print(f"  Test set:      {X_test.shape[0]:,} flights   "
      f"(delay rate: {y_test.mean()*100:.1f}%)")
print()


# =============================================================================
# STEP 4: HANDLE CLASS IMBALANCE WITH SMOTE + FEATURE SCALING
# =============================================================================
# TWO separate problems are addressed here:
#
# PROBLEM A — CLASS IMBALANCE
# Only ~21% of flights are delayed. If we train on this imbalanced data,
# a model can achieve 79% accuracy by simply predicting "on time" for every
# flight — which is completely useless. We need to help the model learn to
# identify the minority (delayed) class.
#
# SOLUTION — SMOTE (Synthetic Minority Over-sampling Technique)
# SMOTE creates synthetic delayed-flight examples by interpolating between
# real delayed flights in feature space. For example, it might take two
# real delayed flights and generate a new synthetic example that sits
# between them. This is much better than simply duplicating rows (which
# causes overfitting) or under-sampling the majority class (which discards
# useful data).
#
# CRITICAL RULE: SMOTE is applied ONLY to training data.
# The test set must remain untouched with the original class distribution,
# because that distribution reflects reality. Applying SMOTE to the test
# set would make evaluation metrics misleading.
#
# PROBLEM B — FEATURE SCALING
# Logistic Regression and MLP are sensitive to the scale of features.
# Consider: DISTANCE ranges from 100 to 4,000 miles, while is_weekend is
# just 0 or 1. Without scaling, DISTANCE would dominate the model and
# is_weekend would be treated as nearly irrelevant, even if day-of-week
# is highly predictive.
#
# SOLUTION — StandardScaler (z-score normalisation)
# Each feature is transformed to: (value - mean) / std deviation
# This centres every feature around 0 with unit variance.
#
# CRITICAL RULE: The scaler is FIT on training data only, then APPLIED
# to both train and test. If we fit on the test set, we would "leak"
# future information (future mean and std) into our model — another
# form of data leakage.
#
# RF and XGBoost do not require scaling (tree splits are based on rank
# order, not magnitude), but we scale all features uniformly so that
# our comparison is consistent and our report is clean.
# =============================================================================

print("=" * 65)
print("STEP 4: SMOTE (TRAINING ONLY) + FEATURE SCALING")
print("=" * 65)

# --- 4a. Apply SMOTE to training data only ---
print("  Applying SMOTE to training data...")
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"  Training set before SMOTE: {len(y_train):,} samples  "
      f"(delayed: {y_train.sum():,}  |  on-time: {(y_train==0).sum():,})")
print(f"  Training set after SMOTE:  {len(y_train_sm):,} samples  "
      f"(delayed: {y_train_sm.sum():,}  |  on-time: {(y_train_sm==0).sum():,})")
print()

# --- 4b. Scale features ---
# Fit the scaler on the SMOTE-resampled training data, then apply to test
print("  Fitting StandardScaler on resampled training data...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled  = scaler.transform(X_test)   # use the SAME scaler, do NOT refit

print("  Scaling complete.")
print()


# =============================================================================
# STEP 5: TRAIN ALL FOUR MODELS
# =============================================================================
# We train all four classifiers using the same scaled, SMOTE-resampled
# training data so comparisons are fair.
#
# MODEL RATIONALE (connecting to lecture content):
#
# 1. LOGISTIC REGRESSION (W6 baseline)
#    The lecture introduces LR as the simplest probabilistic classifier —
#    a sigmoid function applied to a linear combination of features.
#    We use L2 (Ridge) regularisation as discussed in W6 and W8 to prevent
#    overfitting on our 30+ feature space. C=1.0 is the inverse regularisation
#    strength (higher C = less regularisation).
#    max_iter=2000 ensures convergence on our large dataset.
#
# 2. RANDOM FOREST (W4 — bagging ensemble)
#    The lecture describes RF as: bootstrap sample → train DT → aggregate
#    by majority vote. We use 300 trees, which is enough to stabilise the
#    ensemble without excessive training time. n_jobs=-1 uses all CPU cores.
#    RF does not need class_weight because we've already balanced with SMOTE.
#
# 3. XGBOOST (W4 — boosting ensemble)
#    The lecture formula: ŷ_new = ŷ_old + η·f(x)
#    Each tree corrects the residual errors of all previous trees.
#    learning_rate=0.1 (η in the lecture formula) controls step size.
#    n_estimators=300 trees, max_depth=6 limits overfitting.
#    use_label_encoder=False and eval_metric='logloss' suppress warnings.
#
# 4. MLP NEURAL NETWORK (W8 — representation learning)
#    The lecture shows MLPClassifier with hidden_layer_sizes=(100,) and
#    activation='relu', solver='adam'. We add a second hidden layer (50
#    neurons) to capture the more complex non-linear interactions between
#    weather, time, and airline features. max_iter=500 allows convergence.
#    early_stopping=True prevents overfitting by stopping when validation
#    loss stops improving (as discussed in W8 epochs section).
# =============================================================================

print("=" * 65)
print("STEP 5: TRAINING ALL FOUR MODELS")
print("=" * 65)

models = {
    "Logistic Regression": LogisticRegression(
        C=1.0,
        max_iter=2000,
        random_state=42,
        solver="lbfgs",
        n_jobs=-1
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,       # let trees grow fully (SMOTE handles imbalance)
        min_samples_leaf=5,   # prevent overly specific leaves
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.1,    # η in the lecture formula
        max_depth=6,
        subsample=0.8,        # use 80% of data per tree to reduce overfitting
        colsample_bytree=0.8, # use 80% of features per tree
        random_state=42,
        eval_metric="logloss",
        verbosity=0,
        n_jobs=-1
    ),
    "MLP Neural Network": MLPClassifier(
        hidden_layer_sizes=(100, 50),  # two hidden layers: 100 then 50 neurons
        activation="relu",             # ReLU: default for hidden layers (W8)
        solver="adam",                 # adam: stochastic gradient optimiser (W8)
        alpha=0.001,                   # L2 regularisation to prevent overfitting
        max_iter=500,
        early_stopping=True,           # stop if validation loss stops improving
        validation_fraction=0.1,       # use 10% of training data for early stopping
        random_state=42
    ),
}

trained_models = {}
for name, model in models.items():
    print(f"  Training {name}...", end="  ", flush=True)
    model.fit(X_train_scaled, y_train_sm)
    trained_models[name] = model
    print("done.")

print()


# =============================================================================


# =============================================================================
# STEP 5b: HYPERPARAMETER TUNING — RANDOM FOREST & XGBOOST
# =============================================================================
# Step 5 trained all four models with sensible default hyperparameters.
# Here we tune Random Forest and XGBoost — the two strongest performers —
# using RandomizedSearchCV, which is the standard approach for large datasets
# where an exhaustive GridSearch would be computationally prohibitive.
#
# HOW RandomizedSearchCV WORKS:
#   Rather than testing every combination of hyperparameters (as GridSearchCV
#   does), RandomizedSearchCV samples n_iter random combinations from the
#   specified distributions. This gives a good approximation of the best
#   configuration at a fraction of the compute cost.
#
#   Total fits = n_iter × cv_folds = 20 × 3 = 60 fits per model.
#   Compared to a full GridSearch over 4 params × 4 values × 3 folds = 768
#   fits, this saves ~92% of compute time with minimal loss in quality.
#
# OUR SEARCH GRIDS:
#  Random Forest:  n_estimators, max_depth, max_features, min_samples_leaf
#  XGBoost:        n_estimators, max_depth, learning_rate, subsample
#
# SCORING: 'f1' — balances precision and recall for our imbalanced classes.
#
# After tuning, trained_models is updated so all downstream steps
# (evaluation, confusion matrices, ROC curves, threshold analysis)
# automatically use the tuned models.
# =============================================================================

print("=" * 65)
print("STEP 5b: HYPERPARAMETER TUNING")
print("=" * 65)
print()
print("  Strategy: RandomizedSearchCV — 20 combinations × 3-fold CV = 60 fits per model")
print("  This is ~92% faster than a full grid search (768 fits) with minimal quality loss.")
print("  Scoring metric: F1 (balances precision and recall for imbalanced data).")
print()

# ── RANDOM FOREST SEARCH ───────────────────────────────────────────────────
rf_param_grid = {
    "n_estimators":     [100, 200, 300, 500],
    "max_depth":        [None, 10, 20, 30],
    "max_features":     ["sqrt", "log2", 0.3],
    "min_samples_leaf": [1, 5, 10],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=rf_param_grid,
    n_iter=20,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=0,
)

print("  Tuning Random Forest...", end="  ", flush=True)
rf_search.fit(X_train_scaled, y_train_sm)
print("done.")
print(f"  Best RF params:   {rf_search.best_params_}")
print(f"  Best RF F1 (CV):  {rf_search.best_score_:.4f}")
print()

# ── XGBOOST SEARCH ────────────────────────────────────────────────────────
xgb_param_grid = {
    "n_estimators":  [100, 200, 300, 500],
    "max_depth":     [4, 6, 8, 10],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample":     [0.7, 0.8, 0.9, 1.0],
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(
        random_state=42, eval_metric="logloss",
        verbosity=0, n_jobs=-1
    ),
    param_distributions=xgb_param_grid,
    n_iter=20,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=0,
)

print("  Tuning XGBoost...", end="  ", flush=True)
xgb_search.fit(X_train_scaled, y_train_sm)
print("done.")
print(f"  Best XGB params:  {xgb_search.best_params_}")
print(f"  Best XGB F1 (CV): {xgb_search.best_score_:.4f}")
print()

# ── REPLACE TRAINED MODELS WITH TUNED VERSIONS ────────────────────────────
trained_models["Random Forest"] = rf_search.best_estimator_
trained_models["XGBoost"]       = xgb_search.best_estimator_

print("  ✓ trained_models['Random Forest'] updated with tuned estimator")
print("  ✓ trained_models['XGBoost'] updated with tuned estimator")
print("  Logistic Regression and MLP remain as trained in Step 5.")
print()

# STEP 6: EVALUATE — METRICS COMPARISON TABLE
# =============================================================================
# We evaluate every model on the ORIGINAL (un-SMOTE'd) test set.
# The test set must reflect the real world: ~21% delayed flights.
#
# METRICS EXPLAINED (connecting to W3 lecture):
#
# Accuracy  = (TP + TN) / total
#             Overall fraction correct — misleading for imbalanced classes
#
# Precision = TP / (TP + FP)
#             Of flights predicted delayed, how many truly were?
#             Low precision → passengers get unnecessary alerts (annoying)
#
# Recall    = TP / (TP + FN)   [also called Sensitivity or TPR in W3]
#             Of flights that were delayed, how many did we catch?
#             Lower recall → fewer delays detected (passengers not warned)
#
# Precision = TP / (TP + FP)
#             Of flights we PREDICTED as delayed, how many actually were?
#             THIS IS OUR PRIMARY METRIC. A false positive (telling a
#             passenger they're delayed when they are not) could cause
#             them to slow down their journey to the airport and miss
#             a flight that actually departed on time. The cost of a
#             false alarm is therefore higher than the cost of a missed
#             delay (the passenger experiences the delay without warning,
#             but does not miss their flight).
#
# F1 Score  = 2 × (Precision × Recall) / (Precision + Recall)
#             Harmonic mean of precision and recall — balances both
#
# AUC       = Area Under ROC Curve (W3)
#             Probability that model ranks a random delayed flight higher
#             than a random on-time flight. Threshold-independent.
#             Perfect model = 1.0, random model = 0.5
#
# We use the DEFAULT 0.5 probability threshold here. In Step 10 we tune
# the threshold to find the operating point that achieves precision ≥ 0.50
# (our operational target) with the highest recall possible.
# =============================================================================

print("=" * 65)
print("STEP 6: MODEL EVALUATION — METRICS TABLE")
print("=" * 65)

results = {}
for name, model in trained_models.items():
    y_pred  = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)

    results[name] = {
        "Accuracy":  accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall":    recall_score(y_test, y_pred),
        "F1 Score":  f1_score(y_test, y_pred),
        "AUC":       roc_auc,
        "y_pred":    y_pred,
        "y_proba":   y_proba,
        "fpr":       fpr,
        "tpr":       tpr,
    }

# Print a formatted comparison table
print()
header = f"  {'Model':<25} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8} {'AUC':>8}"
print(header)
print("  " + "-" * 73)
for name, m in results.items():
    print(f"  {name:<25} {m['Accuracy']:>9.4f} {m['Precision']:>10.4f} "
          f"{m['Recall']:>8.4f} {m['F1 Score']:>8.4f} {m['AUC']:>8.4f}")
print()

# Identify best model by precision (our primary metric) and by PR-AUC
best_precision_model = max(results, key=lambda n: results[n]["Precision"])
best_auc_model       = max(results, key=lambda n: results[n]["AUC"])
print(f"  Best model by Precision: {best_precision_model}")
print(f"  Best model by AUC:       {best_auc_model}")
print()


# =============================================================================
# STEP 7: CONFUSION MATRICES
# =============================================================================
# A confusion matrix shows the four possible prediction outcomes for each
# model. For our binary problem:
#
#   True Negative  (TN): Predicted on-time, actually on-time  ✓ correct
#   False Positive (FP): Predicted delayed, actually on-time  ✗ wrong alert
#   False Negative (FN): Predicted on-time, actually delayed  ✗ missed delay
#   True Positive  (TP): Predicted delayed, actually delayed  ✓ correct
#
# FN is our biggest concern (passengers told "on time" but flight is delayed).
# We want FN to be as small as possible → maximise Recall.
#
# We plot all four models side by side so the report can display them as
# a single figure (mirroring the AquaSafe and CHD example reports).
# =============================================================================

print("=" * 65)
print("STEP 7: CONFUSION MATRICES")
print("=" * 65)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle("Confusion Matrices — Test Set (2023)", fontsize=14, y=1.02)

for ax, (name, m) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, m["y_pred"])
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", ax=ax,
        xticklabels=["Pred: On-time", "Pred: Delayed"],
        yticklabels=["Act: On-time",  "Act: Delayed"],
        linewidths=0.5
    )
    ax.set_title(name, fontsize=11, fontweight="bold")
    ax.set_xlabel("Predicted", fontsize=9)
    ax.set_ylabel("Actual",    fontsize=9)

plt.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, "confusion_matrices.png"), dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: confusion_matrices.png")
print()


# =============================================================================
# STEP 8: ROC CURVES
# =============================================================================
# The ROC (Receiver Operating Characteristic) curve plots:
#   x-axis: False Positive Rate (1 - Specificity) — proportion of on-time
#           flights incorrectly flagged as delayed
#   y-axis: True Positive Rate (Recall/Sensitivity) — proportion of truly
#           delayed flights that we correctly caught
#
# The curve is drawn by varying the probability threshold from 0 to 1.
# At threshold=0, every flight is predicted delayed (TPR=1, FPR=1).
# At threshold=1, every flight is predicted on-time (TPR=0, FPR=0).
#
# AUC = Area Under the Curve:
#   AUC = 1.0 → perfect classifier
#   AUC = 0.5 → random guessing (the diagonal dashed line)
#   AUC = 0.0 → perfectly wrong
#
# We overlay all four models on one plot so the report can make direct
# visual comparisons between models, exactly as the AquaSafe report did
# (their Figure 8 and Figure 10).
# =============================================================================

print("=" * 65)
print("STEP 8: ROC CURVES")
print("=" * 65)

fig, ax = plt.subplots(figsize=(8, 7))

colours = ["#2196F3", "#4CAF50", "#FF5722", "#9C27B0"]
for (name, m), colour in zip(results.items(), colours):
    ax.plot(m["fpr"], m["tpr"], lw=2, color=colour,
            label=f"{name}  (AUC = {m['AUC']:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=1.5, label="Random classifier (AUC = 0.500)")
ax.set_xlabel("False Positive Rate (1 − Specificity)", fontsize=12)
ax.set_ylabel("True Positive Rate (Recall / Sensitivity)", fontsize=12)
ax.set_title("ROC Curves — All Models on 2023 Test Set", fontsize=13, fontweight="bold")
ax.legend(loc="lower right", fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, "roc_curves.png"), dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: roc_curves.png")
print()


# =============================================================================
# STEP 8b: PRECISION-RECALL CURVES
# =============================================================================
# The ROC curve (Step 8) plots TPR vs FPR and is useful for comparing
# models generally, but it does not directly show precision — our primary
# metric. The Precision-Recall (PR) curve is the more appropriate diagnostic
# when precision matters more than recall.
#
# PR CURVE EXPLAINED:
#   x-axis: Recall (TPR) — proportion of true delays caught
#   y-axis: Precision    — proportion of delay alerts that are genuine
#
# The curve is swept by varying the probability threshold from 0 to 1:
#   At low threshold → high recall, low precision (many alerts, many false)
#   At high threshold → low recall, high precision (few alerts, mostly true)
#
# PR-AUC = Area Under the Precision-Recall Curve
#   A perfect classifier = PR-AUC of 1.0
#   A random classifier = PR-AUC ≈ class prevalence (here ~0.228)
#   PR-AUC is more informative than ROC-AUC for imbalanced datasets,
#   because it explicitly penalises false positives — which matter most
#   in our passenger-facing use case.
#
# The no-skill baseline (horizontal dashed line) represents a classifier
# that always predicts the majority class. Its precision equals the
# positive class rate in the test set (~22.8%).
# =============================================================================

print("=" * 65)
print("STEP 8b: PRECISION-RECALL CURVES")
print("=" * 65)

from sklearn.metrics import precision_recall_curve, average_precision_score

fig, ax = plt.subplots(figsize=(8, 7))

colours = ["#2196F3", "#4CAF50", "#FF5722", "#9C27B0"]
for (name, m), colour in zip(results.items(), colours):
    prec_curve, rec_curve, _ = precision_recall_curve(y_test, m["y_proba"])
    pr_auc = average_precision_score(y_test, m["y_proba"])
    ax.plot(rec_curve, prec_curve, lw=2, color=colour,
            label=f"{name}  (PR-AUC = {pr_auc:.3f})")

# No-skill baseline: a classifier that always predicts positive
# achieves precision = positive class prevalence
no_skill_precision = y_test.mean()
ax.axhline(no_skill_precision, color="black", linestyle="--", lw=1.5,
           label=f"No-skill baseline (precision = {no_skill_precision:.3f})")

# Mark the precision target as a horizontal reference line
ax.axhline(PRECISION_TARGET, color="blue", linestyle=":", lw=1.5, alpha=0.7,
           label=f"Precision target = {PRECISION_TARGET}")

ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title("Precision-Recall Curves — All Models on 2023 Test Set",
             fontsize=13, fontweight="bold")
ax.legend(loc="upper right", fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, "precision_recall_curves.png"), dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: precision_recall_curves.png")
print()



# =============================================================================
# STEP 9: FEATURE IMPORTANCE
# =============================================================================
# Feature importance tells us WHICH input variables are most influential
# in predicting flight delays. This is one of the most valuable outputs
# for your report's findings and discussion sections.
#
# FOR RANDOM FOREST:
# sklearn calculates feature importance as the total reduction in Gini
# impurity across all splits that used each feature, averaged over all trees.
# Features used early in deep trees with many samples contribute more.
# This is the "mean decrease in impurity" method.
#
# FOR XGBOOST:
# XGBoost calculates importance as "gain" — the average improvement in
# the loss function contributed by splits on each feature. This is
# arguably more meaningful than the RF Gini method because it directly
# reflects how much each feature reduces prediction error.
#
# We plot the top 20 features for each model. Comparing the two rankings
# is itself an interesting finding — RF and XGBoost may rank features
# differently because they explore the feature space differently.
# =============================================================================

print("=" * 65)
print("STEP 9: FEATURE IMPORTANCE")
print("=" * 65)

TOP_N = 20

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle(f"Top {TOP_N} Feature Importances", fontsize=14, fontweight="bold")

importance_data = {}

for ax, model_name in zip(axes, ["Random Forest", "XGBoost"]):
    model = trained_models[model_name]
    importances = model.feature_importances_
    fi_df = pd.DataFrame({
        "Feature":    feature_cols,
        "Importance": importances
    }).sort_values("Importance", ascending=False).head(TOP_N)

    importance_data[model_name] = fi_df

    sns.barplot(
        data=fi_df, x="Importance", y="Feature",
        ax=ax, palette="viridis"
    )
    ax.set_title(f"{model_name}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Importance Score", fontsize=10)
    ax.set_ylabel("")

plt.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, "feature_importance.png"), dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: feature_importance.png")

# Print top 10 features for each model
for model_name, fi_df in importance_data.items():
    print(f"\n  Top 10 features — {model_name}:")
    for i, row in enumerate(fi_df.head(10).itertuples(), 1):
        print(f"    {i:2}. {row.Feature:<30}  {row.Importance:.4f}")
print()


# =============================================================================
# STEP 10: THRESHOLD ANALYSIS
# =============================================================================
# All the metrics in Step 6 used the default 0.5 probability threshold.
# But 0.5 is not necessarily the best operating point for our use case.
#
# Our PRIMARY METRIC is PRECISION (avoid false alarms). The motivation is
# that a false positive — telling a passenger their flight is delayed when
# it is not — could cause the passenger to slow their journey to the
# airport and miss a flight that actually departs on time. We therefore
# target an operational precision of >= 0.50, meaning that when the system
# issues a delay alert, the alert should be correct at least half the time.
#
# Raising the threshold (e.g., to 0.65) means we only predict "delayed"
# when the model is highly confident → precision rises, but recall falls
# (most genuine delays will be missed). This is acceptable here because
# we prefer fewer, more reliable alerts over many noisy ones.
#
# We search a wide threshold range (0.10–0.95) so we capture the
# high-confidence operating region where the precision target is met,
# and plot the precision-recall tradeoff for all four models.
# =============================================================================

print("=" * 65)
print("STEP 10: THRESHOLD ANALYSIS (PRECISION-FIRST)")
print("=" * 65)

# Extended range to capture high-precision operating points
thresholds = np.arange(0.10, 0.96, 0.05)
PRECISION_TARGET = 0.50   # operational precision target

# Plot tradeoff for all four models (2x2 grid)
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("Threshold Analysis — Precision / Recall Tradeoff", fontsize=13)
axes_flat = axes.flatten()

threshold_summary = {}

for ax, (name, m) in zip(axes_flat, results.items()):
    precisions, recalls, f1s = [], [], []
    for thresh in thresholds:
        y_pred_t = (m["y_proba"] >= thresh).astype(int)
        precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
        recalls.append(recall_score(y_test, y_pred_t))
        f1s.append(f1_score(y_test, y_pred_t))
    threshold_summary[name] = {
        "thresholds": thresholds,
        "precision":  precisions,
        "recall":     recalls,
        "f1":         f1s,
    }
    ax.plot(thresholds, precisions, "b-o", ms=5, label="Precision")
    ax.plot(thresholds, recalls,    "r-s", ms=5, label="Recall")
    ax.plot(thresholds, f1s,        "g-^", ms=5, label="F1 Score")
    ax.axhline(PRECISION_TARGET, color="blue", linestyle="--", alpha=0.6,
               label=f"Precision target = {PRECISION_TARGET} (minimum for operational usefulness)")
    ax.set_title(name, fontweight="bold")
    ax.set_xlabel("Probability Threshold")
    ax.set_ylabel("Score")
    ax.legend(fontsize=9, loc="center right")
    ax.set_ylim([0, 1.05])
    ax.grid(alpha=0.3)

# Find threshold for each model that achieves precision >= target with
# the highest recall (i.e. the most useful operating point that still
# meets the precision requirement). For models where no threshold in
# range achieves the precision target, flag and report max precision.
print(f"\n  Threshold achieving Precision >= {PRECISION_TARGET} with highest Recall:\n")
all_thresh_results = {}
for name, m in results.items():
    best_thresh, best_f1, best_prec, best_rec = None, 0, 0, 0
    target_met = False
    for thresh in thresholds:
        y_pred_t = (m["y_proba"] >= thresh).astype(int)
        rec  = recall_score(y_test, y_pred_t)
        prec = precision_score(y_test, y_pred_t, zero_division=0)
        f1   = f1_score(y_test, y_pred_t)
        # Among thresholds meeting the precision target, pick the one
        # with the highest recall (== lowest threshold that still meets it)
        if prec >= PRECISION_TARGET and rec > best_rec:
            best_thresh, best_f1, best_prec, best_rec = thresh, f1, prec, rec
            target_met = True

    if not target_met:
        # No threshold in range meets precision target — report max-precision point
        max_prec_idx = int(np.argmax(threshold_summary[name]["precision"]))
        best_thresh = float(thresholds[max_prec_idx])
        best_prec   = threshold_summary[name]["precision"][max_prec_idx]
        best_rec    = threshold_summary[name]["recall"][max_prec_idx]
        best_f1     = threshold_summary[name]["f1"][max_prec_idx]

    all_thresh_results[name] = {
        "threshold":       best_thresh,
        "recall":          best_rec,
        "precision":       best_prec,
        "f1":              best_f1,
        "target_met":      target_met,
    }
    flag = "" if target_met else "  *** PRECISION TARGET NOT MET ***"
    print(f"  {name:<25}  threshold={best_thresh:.2f}  "
          f"precision={best_prec:.3f}  recall={best_rec:.3f}  f1={best_f1:.3f}{flag}")

plt.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, "threshold_analysis.png"), dpi=150, bbox_inches="tight")
plt.close()
print()
print("  Saved: threshold_analysis.png")
print()


# =============================================================================
# STEP 11: DUAL REGRESSION — PREDICTING DELAY DURATION
# =============================================================================
# The classification models answer: "Will this flight be delayed?"
# The regression model answers:    "If delayed, by HOW MANY MINUTES?"
#
# We run TWO separate Random Forest Regressors to directly measure the
# impact of feature selection on delay duration prediction:
#
#   Model A — OPERATIONAL (25 features):
#     The 10 classification features plus 15 additional operational signals
#     that are hypothesised to help predict delay DURATION specifically.
#     These include congestion signals (hourly_dep_count), destination airport,
#     discrete adverse weather flags (is_freezing, is_severe_weather), and
#     peak travel period indicators (is_thanksgiving_window, is_summer_peak).
#     The key insight is that duration is driven by RECOVERY CAPACITY —
#     how quickly an airline can reassign crew, de-ice, or source a gate —
#     not just by whether conditions are bad.
#
#   Model B — REDUCED (10 features):
#     The same 10 features used by the classifiers, for a fair comparison.
#     Expected to perform worse: weather and calendar features predict
#     WHETHER a flight delays well, but have less signal for HOW LONG.
#
# EVALUATION METRICS:
#   MAE  — Mean Absolute Error (minutes): average prediction error.
#           Most interpretable: "our model is off by X minutes on average."
#   RMSE — Root Mean Squared Error: penalises large errors more heavily.
#   R²   — Coefficient of determination: proportion of variance explained.
#           R² = 1.0 is perfect; R² = 0 means no better than predicting mean.
#
# BASELINE: always predict the mean training delay. Any useful model beats this.
# =============================================================================

print("=" * 65)
print("STEP 11: DUAL REGRESSION — DELAY DURATION PREDICTION")
print("=" * 65)

from sklearn.metrics import r2_score

# ── DEFINE THE 25-FEATURE OPERATIONAL SET ─────────────────────────────────
# The 10 classification features plus 15 additional operational signals.
# All chosen to be non-redundant with each other (no pair |r| > 0.70).
# Features checked for existence to handle any dataset version differences.

REG_FEATURES_25_CANDIDATES = [
    # ── Base 10 (same as classifiers) ──
    "PRCP",              # precipitation — primary weather delay driver
    "AWND",              # wind speed — independent weather axis
    "SNOW",              # snowfall — distinct from rain; major ORD delay cause
    "avg_temp",          # temperature — TMIN/TMAX dropped (r > 0.95)
    "CRS_DEP_TIME",      # scheduled departure time — peak-hour congestion
    "day_of_year",       # seasonality — month/quarter dropped (r > 0.96)
    "is_weekend",        # weekend vs weekday traffic pattern
    "daily_dep_count",   # total ORD departures that day — congestion proxy
    "DISTANCE",          # route length — CRS_ELAPSED_TIME dropped (r = 0.99)
    "AIRLINE_CODE",      # carrier — on-time performance varies by airline
    # ── Additional operational signals ──
    "hourly_dep_count",  # hourly congestion — finer than daily, helps recovery timing
    "DEST",              # destination airport — hubs recover faster than regionals
    "CRS_ARR_TIME",      # scheduled arrival slot — gate availability at destination
    "FL_NUMBER",         # flight number — proxy for aircraft rotation patterns
    "SNWD",              # snow depth — accumulated ice multiplies de-icing time
    "is_freezing",       # binary freeze flag — de-icing adds predictable minutes
    "is_severe_weather", # composite adverse weather — storms extend delays
    "is_federal_holiday",# holiday demand — high pax loads slow boarding
    "is_thanksgiving_window", # Thanksgiving surge — worst delay cascade of year
    "is_morning_peak",   # morning bank — least buffer, delays compound fastest
    "is_summer_peak",    # summer thunderstorm season at ORD
    "is_day_before_holiday", # pre-holiday: highest passenger volume
    "day_of_week",       # finer than is_weekend — Mon/Fri differ from mid-week
    "TMAX",              # max temperature — extreme heat affects aircraft performance
    "distance_category", # route type — short/medium/long-haul recovery differs
]

# Only keep features that actually exist in the dataframe
REG_FEATURES_25 = [f for f in REG_FEATURES_25_CANDIDATES if f in df.columns]
REG_FEATURES_10 = FINAL_FEATURES   # the 10 classification features

print(f"  Operational feature set:  {len(REG_FEATURES_25)} features")
print(f"  Reduced feature set:      {len(REG_FEATURES_10)} features")
print()

# ── PREPARE DATA ──────────────────────────────────────────────────────────
# We train on REAL delayed flights only (DEP_DELAY >= 15 min).
# SMOTE rows are excluded — they are synthetic and have no meaningful
# delay duration to predict.

train_delayed_mask = y_reg_train >= 15
test_delayed_mask  = y_reg_test  >= 15

y_reg_train_d = y_reg_train[train_delayed_mask]
y_reg_test_d  = y_reg_test[test_delayed_mask]

print(f"  Training on {train_delayed_mask.sum():,} delayed training flights")
print(f"  Testing  on {test_delayed_mask.sum():,}  delayed test flights")
print()

# ── MODEL A: 25-FEATURE OPERATIONAL REGRESSOR ─────────────────────────────
# Requires its own scaler (different feature set from the classifiers)
scaler_reg25 = StandardScaler()

X_reg25_train_raw = df.loc[df["year"] != TEST_YEAR, REG_FEATURES_25].copy().astype(float)
X_reg25_test_raw  = df.loc[df["year"] == TEST_YEAR, REG_FEATURES_25].copy().astype(float)

# Fill any nulls with column median (defensive)
for col in REG_FEATURES_25:
    med = X_reg25_train_raw[col].median()
    X_reg25_train_raw[col] = X_reg25_train_raw[col].fillna(med)
    X_reg25_test_raw[col]  = X_reg25_test_raw[col].fillna(med)

X_reg25_train_scaled = scaler_reg25.fit_transform(X_reg25_train_raw)
X_reg25_test_scaled  = scaler_reg25.transform(X_reg25_test_raw)

X_reg25_train = X_reg25_train_scaled[train_delayed_mask]
X_reg25_test  = X_reg25_test_scaled[test_delayed_mask]

rf_reg_25 = RandomForestRegressor(
    n_estimators=200, max_depth=15,
    min_samples_leaf=5, random_state=42, n_jobs=-1
)
print("  Training RF Regressor — 25 operational features...", end="  ", flush=True)
rf_reg_25.fit(X_reg25_train, y_reg_train_d)
print("done.")
y_pred_25 = rf_reg_25.predict(X_reg25_test)

# ── MODEL B: 10-FEATURE REDUCED REGRESSOR ─────────────────────────────────
# Re-uses the classifier scaler (same 10 features, already scaled)
X_train_orig_scaled = scaler.transform(X_train)   # original rows, no SMOTE

X_reg10_train = X_train_orig_scaled[train_delayed_mask]
X_reg10_test  = X_test_scaled[test_delayed_mask]

rf_reg_10 = RandomForestRegressor(
    n_estimators=200, max_depth=15,
    min_samples_leaf=5, random_state=42, n_jobs=-1
)
print("  Training RF Regressor — 10 reduced features...", end="  ", flush=True)
rf_reg_10.fit(X_reg10_train, y_reg_train_d)
print("done.")
y_pred_10 = rf_reg_10.predict(X_reg10_test)

# ── BASELINE ──────────────────────────────────────────────────────────────
baseline_pred = np.full(len(y_reg_test_d), fill_value=y_reg_train_d.mean())

# ── METRICS ───────────────────────────────────────────────────────────────
def reg_metrics(y_true, y_pred):
    mae  = np.mean(np.abs(y_pred - y_true))
    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
    r2   = r2_score(y_true, y_pred)
    return mae, rmse, r2

mae_25,  rmse_25,  r2_25  = reg_metrics(y_reg_test_d, y_pred_25)
mae_10,  rmse_10,  r2_10  = reg_metrics(y_reg_test_d, y_pred_10)
mae_base,rmse_base,r2_base= reg_metrics(y_reg_test_d, baseline_pred)

print()
print(f"  {'Metric':<10} {'25-feature':>14} {'10-feature':>14} {'Baseline':>12}")
print(f"  {'-'*54}")
print(f"  {'MAE (min)':<10} {mae_25:>14.1f} {mae_10:>14.1f} {mae_base:>12.1f}")
print(f"  {'RMSE (min)':<10} {rmse_25:>14.1f} {rmse_10:>14.1f} {rmse_base:>12.1f}")
print(f"  {'R²':<10} {r2_25:>14.4f} {r2_10:>14.4f} {r2_base:>12.4f}")
print()
print(f"  MAE improvement — 25-feat vs baseline: {((mae_base-mae_25)/mae_base)*100:+.1f}%")
print(f"  MAE improvement — 10-feat vs baseline: {((mae_base-mae_10)/mae_base)*100:+.1f}%")
print(f"  MAE improvement — 25-feat vs 10-feat:  {((mae_10-mae_25)/mae_10)*100:+.1f}%")
print()

# ── PLOT: SIDE-BY-SIDE ACTUAL VS PREDICTED ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
max_val = 400

for ax, y_pred, mae, rmse, r2, title, colour in [
    (axes[0], y_pred_25, mae_25, rmse_25, r2_25,
     f"25-Feature Operational Model", "#2196F3"),
    (axes[1], y_pred_10, mae_10, rmse_10, r2_10,
     f"10-Feature Reduced Model",     "#FF5722"),
]:
    ax.scatter(y_reg_test_d, y_pred, alpha=0.12, s=6, color=colour,
               label="Predicted vs Actual")
    ax.plot([0, max_val], [0, max_val], "k--", lw=1.5, label="Perfect prediction")
    ax.set_xlabel("Actual Delay (minutes)", fontsize=11)
    ax.set_ylabel("Predicted Delay (minutes)", fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlim([0, max_val]); ax.set_ylim([0, max_val])
    ax.legend(fontsize=9)
    ax.text(0.05, 0.88,
            f"MAE  = {mae:.1f} min\nRMSE = {rmse:.1f} min\nR²   = {r2:.4f}",
            transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

fig.suptitle("RF Regressor: Actual vs Predicted Delay Duration\n"
             "(Delayed Flights Only — 2023 Test Set)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, "regression_dual_comparison.png"), dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: regression_dual_comparison.png")
print()

# Keep mae_model / rmse_model pointing to the better model for Step 12 summary
mae_model  = mae_25
rmse_model = rmse_25
mae_baseline  = mae_base
rmse_baseline = rmse_base


# =============================================================================
# STEP 12: FINAL SUMMARY FOR REPORT
# =============================================================================
# Print a clean summary table of all key results for easy copy-paste into
# your report's findings section.
# =============================================================================

print("=" * 65)
print("STEP 12: FINAL SUMMARY")
print("=" * 65)

print("\n  ── CLASSIFICATION RESULTS (default threshold = 0.50) ──\n")
print(f"  {'Model':<25} {'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} {'AUC':>6}")
print(f"  {'-'*55}")
for name, m in results.items():
    print(f"  {name:<25} {m['Accuracy']:.3f}  {m['Precision']:.3f}  "
          f"{m['Recall']:.3f}  {m['F1 Score']:.3f}  {m['AUC']:.3f}")

print(f"\n  ── TUNED THRESHOLD (precision >= {PRECISION_TARGET}, maximising recall) ──\n")
print(f"  {'Model':<25} {'Thresh':>7} {'Prec':>7} {'Recall':>7} {'F1':>7} {'Target?':>9}")
print(f"  {'-'*65}")
for name, r in all_thresh_results.items():
    met = "yes" if r["target_met"] else "NOT MET"
    print(f"  {name:<25} {r['threshold']:>7.2f} {r['precision']:>7.3f} "
          f"{r['recall']:>7.3f} {r['f1']:>7.3f} {met:>9}")

print(f"\n  ── REGRESSION (delayed flights only) ──\n")
print(f"  {'Model':<30} {'MAE (min)':>10} {'RMSE (min)':>12} {'R²':>8}")
print(f"  {'-'*64}")
print(f"  {'RF — 25 operational features':<30} {mae_25:>10.1f} {rmse_25:>12.1f} {r2_25:>8.4f}")
print(f"  {'RF — 10 reduced features':<30} {mae_10:>10.1f} {rmse_10:>12.1f} {r2_10:>8.4f}")
print(f"  {'Baseline (mean prediction)':<30} {mae_base:>10.1f} {rmse_base:>12.1f} {r2_base:>8.4f}")

print()
print("  ── SAVED PLOTS ──")
print("  1. confusion_matrices.png")
print("  2. roc_curves.png")
print("  3. precision_recall_curves.png")
print("  4. feature_importance.png")
print("  5. threshold_analysis.png")
print("  6. regression_dual_comparison.png")

print()
print("=" * 65)
print("  PIPELINE COMPLETE")
print("=" * 65)

STEP 1: LOADING DATA
  Dataset shape:       81,452 rows × 78 columns
  Years present:       [np.int64(2019), np.int64(2020), np.int64(2022), np.int64(2023)]
  Delayed flights:     17,388  (21.3%)
  On-time flights:     64,064  (78.7%)

STEP 2: DEFINING FEATURES AND REMOVING LEAKAGE
  Columns excluded (leakage + identifiers): 22
  Feature columns remaining:                58
  (These will be reduced to 10 after multicollinearity analysis in Step 2b)

  Categorical columns detected and encoded: ['AIRLINE', 'AIRLINE_DOT', 'AIRLINE_CODE', 'ORIGIN_CITY', 'DEST', 'DEST_CITY', 'season', 'distance_category']
  Null values remaining after encoding: 0

STEP 2b: MULTICOLLINEARITY ASSESSMENT AND FEATURE SELECTION
  Saved: correlation_matrix.png

  Highly correlated feature pairs (|r| > 0.70):
    |r| = 1.000   AIRLINE  ↔  AIRLINE_DOT
    |r| = 1.000   is_xmas_period  ↔  is_christmas_window
    |r| = 0.999   CRS_ARR_TIME  ↔  arr_hour
    |r| = 0.999   CRS_DEP_TIME  ↔  dep_hour
    |r| = 0.996   mon